# Comprehensive OSFT Training Tutorial

This notebook provides a comprehensive guide to Orthogonal Subspace Fine-Tuning (OSFT) using the Training Hub library. We cover:

- **What OSFT is** and why it prevents catastrophic forgetting
- **All available parameters** with detailed explanations
- **The key `unfreeze_rank_ratio` parameter** — how it works and how to choose a value
- **Single-node and multi-node** distributed configurations
- **Popular model examples** (Qwen 2.5 7B, Llama 3.1 8B, Phi 4 Mini)

OSFT is based on [Nayak et al. (2025), arXiv:2504.07097](https://arxiv.org/abs/2504.07097) and enables continual training **without** catastrophic forgetting and **without** needing replay buffers.

> For a minimal quickstart, see `osft_quickstart.py` in this directory.

## What is OSFT?

Traditional fine-tuning updates all parameters and can overwrite previously learned knowledge. OSFT instead:

1. **Identifies orthogonal subspaces** in the model's weight matrices
2. **Restricts updates to these subspaces**, preserving existing knowledge
3. **Eliminates the need for replay buffers** or supplementary datasets

### OSFT vs Traditional SFT

| Aspect | SFT | OSFT |
|--------|-----|------|
| Catastrophic forgetting | Common problem | Prevented by design |
| Data requirements | Needs replay/mixed data | Only new domain data |
| Preservation method | Data mixing ratios | Mathematical guarantees |

### When to Use OSFT

- Adding domain-specific knowledge (medical, legal, technical)
- Adapting to new languages or instruction formats
- Continual learning across multiple domains
- Any scenario where you must preserve existing capabilities

## Understanding `unfreeze_rank_ratio`

This is the most important OSFT-specific parameter. It controls how much of each weight matrix can be updated:

- **Range:** 0.0 to 1.0
- **Lower values** → more preservation, slower adaptation
- **Higher values** → more adaptation, slightly less preservation

### Recommended Settings

| Use Case | Ratio | Rationale |
|----------|-------|-----------|
| Minor format adjustments | 0.10–0.15 | Maximum preservation |
| Domain vocabulary addition | 0.15–0.25 | Add terms without losing general knowledge |
| Domain specialization | 0.25–0.35 | Balance preservation and adaptation |
| Major capability expansion | 0.35–0.50 | Significant new learning required |

**Start conservative (0.2–0.3) and increase only if needed.**

## Setup and Imports

In [ ]:
from training_hub import osft

import os
import time
from datetime import datetime
from pathlib import Path

## Data Format Requirements

OSFT uses the same JSONL messages format as SFT:

```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

### Masking Control

Use the `unmask_messages` parameter (not a per-sample field):

- `unmask_messages=False` (default) — train only on assistant turns
- `unmask_messages=True` — train on all content except system messages

### Pre-processed Data

If your data already has `input_ids` and `labels`, use `use_processed_dataset=True`.

## Model Configuration Examples

In [ ]:
MODEL_CONFIGS = {
    "qwen_7b": {
        "model_path": "Qwen/Qwen2.5-7B-Instruct",
        "unfreeze_rank_ratio": 0.25,
        "max_tokens_per_gpu": 10_000,
        "max_seq_len": 8_196,
        "effective_batch_size": 128,
        "learning_rate": 5e-6,
    },
    "llama_8b": {
        "model_path": "meta-llama/Llama-3.1-8B-Instruct",
        "unfreeze_rank_ratio": 0.3,
        "max_tokens_per_gpu": 10_000,
        "max_seq_len": 8_192,
        "effective_batch_size": 128,
        "learning_rate": 5e-6,
    },
    "phi_mini": {
        "model_path": "microsoft/Phi-4-mini-instruct",
        "unfreeze_rank_ratio": 0.25,
        "max_tokens_per_gpu": 8_192,
        "max_seq_len": 4_096,
        "effective_batch_size": 64,
        "learning_rate": 5e-6,
    },
    "small_1b_3b": {
        "model_path": "/path/to/small-model",
        "unfreeze_rank_ratio": 0.4,
        "max_tokens_per_gpu": 16_000,
        "max_seq_len": 4_096,
        "effective_batch_size": 128,
        "learning_rate": 3e-5,
    },
}

# ---- Select your configuration ----
selected = MODEL_CONFIGS["qwen_7b"]

for k, v in selected.items():
    print(f"  {k}: {v}")

## Complete Parameter Configuration

In [ ]:
experiment_name = "osft_comprehensive_example"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Required ----
model_path         = selected["model_path"]
data_path          = "/path/to/your/training_data.jsonl"
ckpt_output_dir    = f"/path/to/checkpoints/{experiment_name}_{timestamp}"
unfreeze_rank_ratio = selected["unfreeze_rank_ratio"]
effective_batch_size = selected["effective_batch_size"]
max_tokens_per_gpu  = selected["max_tokens_per_gpu"]
max_seq_len         = selected["max_seq_len"]
learning_rate       = selected["learning_rate"]

# ---- OSFT-specific ----
target_patterns = None  # None = apply to all layers (recommended)

# ---- Training hyperparameters ----
num_epochs     = 1
seed           = 42
lr_scheduler   = "cosine"
warmup_steps   = 0

# ---- Performance ----
use_liger = True  # Liger kernels for efficiency

# ---- Data processing ----
data_output_dir       = "/dev/shm/osft_data"
use_processed_dataset = False
unmask_messages       = False

# ---- Checkpointing ----
checkpoint_at_epoch    = True
save_final_checkpoint  = True

print(f"Experiment: {experiment_name}_{timestamp}")
print(f"Model:      {model_path}")
print(f"Unfreeze:   {unfreeze_rank_ratio}")

## Distributed Training Configuration

In [ ]:
DIST_PRESETS = {
    "single_gpu": dict(nproc_per_node=1, nnodes=1, node_rank=0,
                        rdzv_id=1, rdzv_endpoint="127.0.0.1:29500"),
    "single_node_8gpu": dict(nproc_per_node=8, nnodes=1, node_rank=0,
                              rdzv_id=100, rdzv_endpoint="127.0.0.1:29500"),
    "multi_node_master": dict(nproc_per_node=8, nnodes=2, node_rank=0,
                               rdzv_id=42, rdzv_endpoint="10.0.0.1:29500"),
    "multi_node_worker": dict(nproc_per_node=8, nnodes=2, node_rank=1,
                               rdzv_id=42, rdzv_endpoint="10.0.0.1:29500"),
}

# ---- Select your setup ----
dist = DIST_PRESETS["single_node_8gpu"]

total_gpus = dist["nproc_per_node"] * dist["nnodes"]
print(f"Total GPUs: {total_gpus}")
print("OSFT works seamlessly across nodes — no replay buffer coordination needed.")

## Execute Training

In [ ]:
training_params = dict(
    # Required
    model_path=model_path,
    data_path=data_path,
    ckpt_output_dir=ckpt_output_dir,
    unfreeze_rank_ratio=unfreeze_rank_ratio,
    effective_batch_size=effective_batch_size,
    max_tokens_per_gpu=max_tokens_per_gpu,
    max_seq_len=max_seq_len,
    learning_rate=learning_rate,
    # OSFT-specific
    target_patterns=target_patterns,
    # Training
    num_epochs=num_epochs,
    seed=seed,
    lr_scheduler=lr_scheduler,
    lr_scheduler_kwargs={},
    warmup_steps=warmup_steps,
    # Performance
    use_liger=use_liger,
    # Data
    data_output_dir=data_output_dir,
    use_processed_dataset=use_processed_dataset,
    unmask_messages=unmask_messages,
    # Checkpointing
    checkpoint_at_epoch=checkpoint_at_epoch,
    save_final_checkpoint=save_final_checkpoint,
    # Distributed
    **dist,
)

print("Training configuration:")
for k, v in training_params.items():
    if v is not None:
        print(f"  {k}: {v}")

start_time = time.time()

try:
    result = osft(**training_params)
    elapsed = time.time() - start_time
    print(f"\nOSFT training completed in {elapsed / 3600:.2f} hours")
    print(f"Checkpoints saved to: {ckpt_output_dir}")
except Exception as exc:
    elapsed = time.time() - start_time
    print(f"\nTraining failed after {elapsed / 60:.1f} minutes: {exc}")
    print("\nTroubleshooting:")
    print("  - Reduce max_tokens_per_gpu for OOM errors")
    print("  - Adjust unfreeze_rank_ratio (lower = less memory)")
    print("  - Verify data_path and model_path are accessible")
    raise

## Post-Training Analysis

In [ ]:
if os.path.isdir(ckpt_output_dir):
    checkpoints = sorted(
        d for d in os.listdir(ckpt_output_dir)
        if os.path.isdir(os.path.join(ckpt_output_dir, d))
    )
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        print(f"  {ckpt}")

    if checkpoints:
        final = os.path.join(ckpt_output_dir, checkpoints[-1])
        print(f"\nFinal checkpoint: {final}")
        print("\nLoad your OSFT-adapted model:")
        print(f"  model = AutoModelForCausalLM.from_pretrained('{final}')")
        print(f"  tokenizer = AutoTokenizer.from_pretrained('{final}')")
else:
    print(f"Checkpoint directory not found: {ckpt_output_dir}")

print(f"\nOSFT Validation Steps:")
print("  1. Test original capabilities — model should still perform well on general tasks")
print("  2. Test new domain — confirm improved performance on target domain")
print("  3. Compare with base model — run side-by-side comparisons")
print(f"  4. If more adaptation needed, increase unfreeze_rank_ratio (currently {unfreeze_rank_ratio})")

## Parameter Reference

### Required Parameters

| Parameter | Description |
|-----------|-------------|
| `model_path` | HuggingFace model ID or local path |
| `data_path` | Path to JSONL training data |
| `ckpt_output_dir` | Checkpoint output directory |
| `unfreeze_rank_ratio` | Fraction of each weight matrix to unfreeze (0.0–1.0) |
| `effective_batch_size` | Global batch size across all GPUs |
| `max_tokens_per_gpu` | Token budget per GPU per step |
| `max_seq_len` | Maximum sequence length |
| `learning_rate` | Optimizer learning rate |

### OSFT-Specific Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `unfreeze_rank_ratio` | — | Controls preservation vs adaptation trade-off |
| `target_patterns` | `None` | Optional layer-name substring patterns (leave `None` for all layers) |

### Training & Performance

| Parameter | Default | Description |
|-----------|---------|-------------|
| `num_epochs` | 1 | Training epochs |
| `seed` | 42 | Random seed |
| `lr_scheduler` | `"cosine"` | LR schedule type |
| `use_liger` | `False` | Use Liger kernels for memory efficiency |
| `unmask_messages` | `False` | Unmask all messages for pretraining-style training |
| `use_processed_dataset` | `False` | Use pre-tokenized data with `input_ids`/`labels` |

### OSFT vs SFT

| Aspect | OSFT | SFT |
|--------|------|-----|
| Catastrophic forgetting | Prevented by design | Requires replay buffers |
| Data requirements | Only new domain data | Needs mixed/replay data |
| Key parameter | `unfreeze_rank_ratio` | N/A |
| Backend | mini-trainer | instructlab-training |
| Best for | Continual learning | Initial fine-tuning |